# Supervised Regime Classifier — Best Config Only


## 1. Imports & Constants

In [ ]:
from pathlib import Path
import warnings
from functools import reduce
from typing import Optional, List, Dict, Tuple, Any

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report
from scipy.stats import ttest_ind, f_oneway, kruskal

warnings.filterwarnings("ignore")

try:
    from lightgbm import LGBMClassifier
    _HAS_LGBM = True
except ImportError:
    _HAS_LGBM = False
    print("LightGBM not found; will use fallback classifier.")

INDEX_PATH = "index_data.csv"
RETURN_PATH = "return.csv"
OUTPUT_PATH = "supervised_full_sample_predictions.csv"

MAIN_CODES = ["000001", "399001", "000020", "000905", "399005", "000852"]
GROWTH_CODES = ["000688", "399006", "399673"]
TRAIN_END = 20241231
ANN_FACTOR = 242
EPS = 1e-12


## 2. Helper Functions

In [ ]:

def rolling_zscore(series: pd.Series, window: int = 252, min_periods: int = 60) -> pd.Series:
    mu = series.rolling(window, min_periods=min_periods).mean()
    sd = series.rolling(window, min_periods=min_periods).std()
    return (series - mu) / sd.replace(0, np.nan)


def calc_max_drawdown(ret_series: pd.Series) -> float:
    if len(ret_series) == 0:
        return np.nan
    nav = (1 + ret_series.fillna(0)).cumprod()
    peak = nav.cummax()
    return (nav / peak - 1).min()


def calc_tstat(x) -> float:
    x = pd.Series(x).dropna()
    if len(x) < 2:
        return np.nan
    s = x.std()
    if pd.isna(s) or s < EPS:
        return np.nan
    return float(x.mean() / (s / np.sqrt(len(x))))


def summarize_strategy_by_state(
    df: pd.DataFrame,
    state_col: str = "state",
    ret_col: str = "ret",
    ann_factor: int = ANN_FACTOR
) -> pd.DataFrame:
    rows = []
    total = len(df[ret_col].dropna()) if ret_col in df.columns else 0
    for s, g in df.groupby(state_col):
        r = g[ret_col].dropna() if ret_col in g.columns else pd.Series(dtype=float)
        if len(r) == 0:
            continue
        mean_r = r.mean()
        std_r = r.std()
        rows.append({
            "state": s,
            "count": len(r),
            "count_pct": len(r) / total if total else np.nan,
            "mean_ret": mean_r,
            "std_ret": std_r,
            "tstat": calc_tstat(r),
            "sharpe_ann": mean_r / std_r * np.sqrt(ann_factor) if std_r > EPS else np.nan,
            "win_rate": (r > 0).mean(),
            "max_drawdown": calc_max_drawdown(r)
        })
    return pd.DataFrame(rows).sort_values("state").reset_index(drop=True)


def state_return_hypothesis_tests(
    df: pd.DataFrame,
    state_col: str = "state",
    ret_col: str = "ret"
) -> Dict[str, Any]:
    work = df[[state_col, ret_col]].dropna().copy()
    states = sorted(work[state_col].unique().tolist())
    groups = [work.loc[work[state_col] == s, ret_col].dropna() for s in states]
    groups = [g for g in groups if len(g) > 1]

    result: Dict[str, Any] = {}
    if len(groups) >= 2:
        try:
            f, p = f_oneway(*groups)
            result["anova_F"] = f
            result["anova_p"] = p
        except Exception:
            result["anova_F"] = np.nan
            result["anova_p"] = np.nan
        try:
            _, p_kw = kruskal(*groups)
            result["kruskal_p"] = p_kw
        except Exception:
            result["kruskal_p"] = np.nan
    else:
        result["anova_F"] = np.nan
        result["anova_p"] = np.nan
        result["kruskal_p"] = np.nan

    pair_rows = []
    for i in range(len(states)):
        for j in range(i + 1, len(states)):
            s1, s2 = states[i], states[j]
            x1 = work.loc[work[state_col] == s1, ret_col].dropna()
            x2 = work.loc[work[state_col] == s2, ret_col].dropna()
            if len(x1) < 2 or len(x2) < 2:
                continue
            t, p = ttest_ind(x1, x2, equal_var=False, nan_policy="omit")
            pair_rows.append({
                "state_i": s1,
                "state_j": s2,
                "mean_i": x1.mean(),
                "mean_j": x2.mean(),
                "mean_diff": x1.mean() - x2.mean(),
                "t_stat": t,
                "p_value": p,
            })
    result["pairwise_tests"] = pd.DataFrame(pair_rows)
    return result


def evaluate_oos_states(oos_df: pd.DataFrame, label: str = "OOS") -> Dict[str, Any]:
    if len(oos_df) == 0:
        print(f"[{label}] no OOS rows")
        return {}

    print(f"[{label}] state counts")
    print(oos_df["state"].value_counts().sort_index())
    print()

    perf = summarize_strategy_by_state(oos_df, state_col="state", ret_col="ret")
    tests = state_return_hypothesis_tests(oos_df, state_col="state", ret_col="ret")
    print(f"[{label}] performance by state")
    display(perf.round(6))
    print(f"[{label}] tests")
    print({
        "anova_p": None if pd.isna(tests["anova_p"]) else round(float(tests["anova_p"]), 6),
        "kruskal_p": None if pd.isna(tests["kruskal_p"]) else round(float(tests["kruskal_p"]), 6),
    })
    if len(tests["pairwise_tests"]) > 0:
        display(tests["pairwise_tests"].round(6))
    return {"performance_by_state": perf, "tests": tests}


## 3. Feature Engineering

In [ ]:

def build_single_index_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy().sort_values("trade_date").reset_index(drop=True)

    df["r1"]  = df["close"] / df["preclose"].replace(0, np.nan) - 1
    df["r3"]  = df["close"] / df["close"].shift(3).replace(0, np.nan) - 1
    df["r5"]  = df["close"] / df["close"].shift(5).replace(0, np.nan) - 1
    df["r10"] = df["close"] / df["close"].shift(10).replace(0, np.nan) - 1
    df["r20"] = df["close"] / df["close"].shift(20).replace(0, np.nan) - 1
    df["r60"] = df["close"] / df["close"].shift(60).replace(0, np.nan) - 1

    df["amp"] = df["high"] / df["low"].replace(0, np.nan) - 1
    df["intraday_ret"] = df["close"] / df["open"].replace(0, np.nan) - 1
    df["gap"] = df["open"] / df["preclose"].replace(0, np.nan) - 1

    df["vol_5"]  = df["r1"].rolling(5).std()
    df["vol_10"] = df["r1"].rolling(10).std()
    df["vol_20"] = df["r1"].rolling(20).std()
    df["vol_60"] = df["r1"].rolling(60).std()

    df["vol_ratio_5_20"]  = df["vol_5"] / df["vol_20"].replace(0, np.nan) - 1
    df["vol_ratio_5_60"]  = df["vol_5"] / df["vol_60"].replace(0, np.nan) - 1
    df["vol_ratio_20_60"] = df["vol_20"] / df["vol_60"].replace(0, np.nan) - 1

    df["ma5"]  = df["close"].rolling(5).mean()
    df["ma20"] = df["close"].rolling(20).mean()
    df["ma60"] = df["close"].rolling(60).mean()

    df["ma5_bias"] = df["close"] / df["ma5"].replace(0, np.nan) - 1
    df["ma20_bias"] = df["close"] / df["ma20"].replace(0, np.nan) - 1
    df["ma60_bias"] = df["close"] / df["ma60"].replace(0, np.nan) - 1
    df["ma20_over_ma60"] = df["ma20"] / df["ma60"].replace(0, np.nan) - 1

    df["high_20d"] = df["close"].rolling(20).max()
    df["low_20d"] = df["close"].rolling(20).min()
    df["dist_from_high_20"] = df["close"] / df["high_20d"].replace(0, np.nan) - 1
    df["dist_from_low_20"] = df["close"] / df["low_20d"].replace(0, np.nan) - 1

    up1 = (df["r1"] > 0).astype(float)
    df["up_frac_5"] = up1.rolling(5).mean()
    df["up_frac_10"] = up1.rolling(10).mean()
    df["up_frac_20"] = up1.rolling(20).mean()

    df["liq_z_20"] = rolling_zscore(df["volume"], 20, 10)
    df["liq_z_60"] = rolling_zscore(df["volume"], 60, 20)
    df["amount_z_20"] = rolling_zscore(df["amount"], 20, 10)
    df["amount_z_60"] = rolling_zscore(df["amount"], 60, 20)

    df["r5_minus_r20"] = df["r5"] - df["r20"]
    df["ma20_bias_chg_5"] = df["ma20_bias"].diff(5)
    df["liq_z_20_chg_5"] = df["liq_z_20"].diff(5)
    df["vol_5_chg_5"] = df["vol_5"].diff(5)

    return df


def build_market_feature_table(df_raw: pd.DataFrame) -> pd.DataFrame:
    pieces = []
    for code_raw, g in df_raw.groupby("idx"):
        x = build_single_index_features(g)
        x["idx"] = str(int(code_raw) % 1_000_000).zfill(6)
        pieces.append(x)

    feat_long = pd.concat(pieces, axis=0, ignore_index=True)
    keep_cols = [
        "trade_date", "idx",
        "r1", "r3", "r5", "r10", "r20", "r60",
        "amp", "intraday_ret", "gap",
        "vol_5", "vol_10", "vol_20",
        "vol_ratio_5_20", "vol_ratio_5_60", "vol_ratio_20_60",
        "ma5_bias", "ma20_bias", "ma60_bias", "ma20_over_ma60",
        "dist_from_high_20", "dist_from_low_20",
        "up_frac_5", "up_frac_10", "up_frac_20",
        "liq_z_20", "liq_z_60", "amount_z_20", "amount_z_60",
        "r5_minus_r20", "ma20_bias_chg_5", "liq_z_20_chg_5", "vol_5_chg_5"
    ]
    feat_long = feat_long[[c for c in keep_cols if c in feat_long.columns]].copy()

    wide = []
    for idx_name, g in feat_long.groupby("idx"):
        gg = g.drop(columns=["idx"]).copy()
        rename_map = {c: f"{idx_name}__{c}" for c in gg.columns if c != "trade_date"}
        wide.append(gg.rename(columns=rename_map))

    feat = reduce(lambda l, r: pd.merge(l, r, on="trade_date", how="outer"), wide)
    feat = feat.sort_values("trade_date").reset_index(drop=True)

    def existing_cols(codes, suffix):
        return [f"{c}__{suffix}" for c in codes if f"{c}__{suffix}" in feat.columns]

    for suffix in ["r1", "r5", "r10", "r20", "vol_5", "vol_20",
                   "ma20_bias", "liq_z_20", "amount_z_20",
                   "up_frac_5", "up_frac_10", "up_frac_20"]:
        mc = existing_cols(MAIN_CODES, suffix)
        gc = existing_cols(GROWTH_CODES, suffix)
        if mc:
            feat[f"main_{suffix}"] = feat[mc].mean(axis=1)
        if gc:
            feat[f"growth_{suffix}"] = feat[gc].mean(axis=1)
        if mc and gc:
            feat[f"style_spread_{suffix}"] = feat[f"main_{suffix}"] - feat[f"growth_{suffix}"]

    base_r1 = "000001__r1" if "000001__r1" in feat.columns else None
    base_r5 = "000001__r5" if "000001__r5" in feat.columns else None
    base_r20 = "000001__r20" if "000001__r20" in feat.columns else None
    for suffix, base in [("r1", base_r1), ("r5", base_r5), ("r20", base_r20)]:
        risk_cols = [c for c in [f"399006__{suffix}", f"000688__{suffix}", f"000852__{suffix}"] if c in feat.columns]
        if base and risk_cols:
            feat[f"risk_on_{suffix}"] = feat[risk_cols].mean(axis=1) - feat[base]

    all_r1_cols = [c for c in feat.columns if c.endswith("__r1")]
    all_r5_cols = [c for c in feat.columns if c.endswith("__r5")]
    all_ma20_cols = [c for c in feat.columns if c.endswith("__ma20_bias")]
    if all_r1_cols:
        feat["breadth_up_ratio_1"] = feat[all_r1_cols].gt(0).mean(axis=1)
        feat["cross_dispersion_r1"] = feat[all_r1_cols].std(axis=1)
    if all_r5_cols:
        feat["breadth_up_ratio_5"] = feat[all_r5_cols].gt(0).mean(axis=1)
        feat["cross_dispersion_r5"] = feat[all_r5_cols].std(axis=1)
    if all_ma20_cols:
        feat["breadth_trend_ratio"] = feat[all_ma20_cols].gt(0).mean(axis=1)

    for code in ["399006", "000688", "000905", "000852", "399001"]:
        for suffix in ["r1", "r5", "r20"]:
            a, b = f"{code}__{suffix}", f"000001__{suffix}"
            if a in feat.columns and b in feat.columns:
                feat[f"{code}_minus_000001_{suffix}"] = feat[a] - feat[b]

    for suffix in ["vol_5", "vol_20"]:
        mc = existing_cols(MAIN_CODES, suffix)
        gc = existing_cols(GROWTH_CODES, suffix)
        if mc and gc:
            feat[f"vol_spread_{suffix}"] = feat[gc].mean(axis=1) - feat[mc].mean(axis=1)

    for col in ["style_spread_r1", "style_spread_r5", "risk_on_r1", "risk_on_r5",
                "breadth_up_ratio_1", "breadth_up_ratio_5", "cross_dispersion_r1",
                "main_ma20_bias", "growth_ma20_bias"]:
        if col in feat.columns:
            feat[f"{col}_chg5"] = feat[col].diff(5)
            feat[f"{col}_chg10"] = feat[col].diff(10)

    vr_cols = existing_cols(MAIN_CODES, "vol_ratio_5_20")
    if vr_cols:
        feat["main_vol_shock"] = feat[vr_cols].mean(axis=1)

    return feat.sort_values("trade_date").reset_index(drop=True)


def select_model_features(feat: pd.DataFrame) -> List[str]:
    preferred = [
        "000001__r1", "000001__r5", "000001__r20", "000001__vol_5", "000001__ma20_bias",
        "000001__liq_z_20", "000001__ma20_over_ma60", "000001__up_frac_10", "000001__up_frac_20",
        "000001__dist_from_high_20",
        "399001__r1", "399001__r5", "399001__r20", "399001__vol_5", "399001__ma20_bias", "399001__liq_z_20",
        "000905__r1", "000905__r5", "000905__r20", "000905__vol_5", "000905__ma20_bias", "000905__liq_z_20",
        "000852__r1", "000852__r5", "000852__r20", "000852__vol_5", "000852__ma20_bias", "000852__liq_z_20",
        "399006__r1", "399006__r5", "399006__r20", "399006__vol_5", "399006__ma20_bias", "399006__liq_z_20",
        "000688__r1", "000688__r5", "000688__r20", "000688__vol_5", "000688__ma20_bias", "000688__liq_z_20",
        "main_r1", "main_r5", "main_r20", "growth_r1", "growth_r5", "growth_r20",
        "style_spread_r1", "style_spread_r5", "style_spread_r20", "style_spread_ma20_bias",
        "risk_on_r1", "risk_on_r5", "risk_on_r20", "breadth_up_ratio_1", "breadth_up_ratio_5",
        "breadth_trend_ratio", "cross_dispersion_r1", "cross_dispersion_r5", "main_ma20_bias",
        "growth_ma20_bias", "main_up_frac_10", "main_up_frac_20", "main_vol_shock",
        "vol_spread_vol_5", "vol_spread_vol_20", "399006_minus_000001_r1", "399006_minus_000001_r5",
        "000688_minus_000001_r1", "000688_minus_000001_r5", "000905_minus_000001_r1", "000905_minus_000001_r5",
        "000852_minus_000001_r1", "000852_minus_000001_r5", "style_spread_r1_chg5", "style_spread_r5_chg5",
        "risk_on_r1_chg5", "risk_on_r5_chg5", "breadth_up_ratio_1_chg5", "breadth_up_ratio_5_chg5",
        "cross_dispersion_r1_chg5", "main_ma20_bias_chg5", "growth_ma20_bias_chg5",
        "style_spread_r1_chg10", "main_ma20_bias_chg10",
    ]
    return [c for c in preferred if c in feat.columns]


## 4. Supervised Learning Pipeline

In [ ]:

def build_return_table(ret_df: pd.DataFrame) -> pd.DataFrame:
    ret_use = ret_df.copy()
    ret_use["trade_date"] = ret_use["date"].astype(int)
    ret_use["ret"] = ret_use["0"].astype(float)
    return ret_use[["trade_date", "ret"]].sort_values("trade_date").reset_index(drop=True)



def build_feature_matrix(
    df_raw: pd.DataFrame,
    feature_cols: Optional[List[str]] = None,
) -> Tuple[pd.DataFrame, List[str]]:
    feat = build_market_feature_table(df_raw).sort_values("trade_date").reset_index(drop=True)
    if feature_cols is None:
        feature_cols = select_model_features(feat)
    else:
        feature_cols = [c for c in feature_cols if c in feat.columns]

    feat = feat.copy()
    feat.loc[:, feature_cols] = feat.loc[:, feature_cols].shift(1)
    return feat, feature_cols



def build_strategy_return_features(ret_use: pd.DataFrame) -> Tuple[pd.DataFrame, List[str]]:
    rs = ret_use[["trade_date", "ret"]].copy().sort_values("trade_date").reset_index(drop=True)
    rs["ret_lag1"] = rs["ret"]
    rs["ret_r3"] = rs["ret"].rolling(3, min_periods=3).sum()
    rs["ret_r5"] = rs["ret"].rolling(5, min_periods=5).sum()
    rs["ret_r10"] = rs["ret"].rolling(10, min_periods=10).sum()
    rs["ret_r20"] = rs["ret"].rolling(20, min_periods=20).sum()
    rs["ret_ewm5"] = rs["ret"].ewm(span=5, adjust=False).mean()
    rs["ret_ewm20"] = rs["ret"].ewm(span=20, adjust=False).mean()
    rs["ret_vol5"] = rs["ret"].rolling(5, min_periods=5).std()
    rs["ret_vol20"] = rs["ret"].rolling(20, min_periods=20).std()
    rs["ret_skew20"] = rs["ret"].rolling(20, min_periods=20).skew()
    rs["ret_win5"] = rs["ret"].gt(0).rolling(5, min_periods=5).mean()
    rs["ret_win20"] = rs["ret"].gt(0).rolling(20, min_periods=20).mean()
    rs["ret_z20"] = rolling_zscore(rs["ret"], 20, 10)
    rs["ret_z60"] = rolling_zscore(rs["ret"], 60, 20)

    nav = (1 + rs["ret"].fillna(0)).cumprod()
    rs["ret_dd20"] = nav / nav.rolling(20, min_periods=5).max() - 1
    rs["ret_dd60"] = nav / nav.rolling(60, min_periods=20).max() - 1
    rs["ret_bad_streak5"] = rs["ret"].lt(0).rolling(5, min_periods=5).sum()

    feature_cols = [
        "ret_lag1", "ret_r3", "ret_r5", "ret_r10", "ret_r20",
        "ret_ewm5", "ret_ewm20", "ret_vol5", "ret_vol20", "ret_skew20",
        "ret_win5", "ret_win20", "ret_z20", "ret_z60",
        "ret_dd20", "ret_dd60", "ret_bad_streak5",
    ]
    rs.loc[:, feature_cols] = rs.loc[:, feature_cols].shift(1)
    return rs[["trade_date"] + feature_cols], feature_cols



def add_strategy_return_features(
    feat_lagged: pd.DataFrame,
    ret_use: pd.DataFrame,
) -> Tuple[pd.DataFrame, List[str]]:
    rs_feat, rs_cols = build_strategy_return_features(ret_use)
    feat = feat_lagged.merge(rs_feat, on="trade_date", how="left")

    interaction_specs = [
        ("risk_on_r5", "ret_r5", "risk_on_x_ret_r5"),
        ("style_spread_r5", "ret_r5", "style_x_ret_r5"),
        ("breadth_up_ratio_5", "ret_dd20", "breadth_x_dd20"),
        ("cross_dispersion_r1", "ret_vol20", "disp_x_ret_vol20"),
        ("main_vol_shock", "ret_dd20", "shock_x_dd20"),
    ]
    interaction_cols: List[str] = []
    for left, right, out_col in interaction_specs:
        if left in feat.columns and right in feat.columns:
            feat[out_col] = feat[left] * feat[right]
            interaction_cols.append(out_col)

    return feat, rs_cols + interaction_cols



def summarize_feature_availability(
    feat_lagged: pd.DataFrame,
    feature_cols: List[str],
) -> Tuple[pd.DataFrame, pd.DataFrame, int]:
    rows = []
    for col in feature_cols:
        first_valid = feat_lagged.loc[feat_lagged[col].notna(), "trade_date"]
        if first_valid.empty:
            continue
        index_code = None
        if "__" in col:
            prefix = col.split("__", 1)[0]
            if prefix.isdigit() and len(prefix) == 6:
                index_code = prefix
        rows.append({
            "feature": col,
            "index_code": index_code,
            "first_valid_date": int(first_valid.iloc[0]),
        })

    feature_start = pd.DataFrame(rows).sort_values(["first_valid_date", "feature"]).reset_index(drop=True)
    if feature_start.empty:
        raise ValueError("No usable feature columns found")

    index_start = (
        feature_start[feature_start["index_code"].notna()]
        .groupby("index_code", as_index=False)
        .agg(
            first_usable_date=("first_valid_date", "max"),
            used_feature_count=("feature", "count"),
        )
        .sort_values(["first_usable_date", "index_code"])
        .reset_index(drop=True)
    )
    model_start_date = int(feature_start["first_valid_date"].max())
    return feature_start, index_start, model_start_date



def fit_label_spec(
    ret_series: pd.Series,
    method: str = "ternary_quantile",
    upper_q: float = 0.67,
    lower_q: float = 0.33,
    upper_thresh: float = 0.006,
    lower_thresh: float = -0.003,
) -> Dict[str, Any]:
    valid = ret_series.dropna()
    if len(valid) == 0:
        raise ValueError("No valid returns for label fitting")
    spec = {
        "method": method,
        "upper_q": upper_q,
        "lower_q": lower_q,
        "upper_thresh": upper_thresh,
        "lower_thresh": lower_thresh,
    }
    if method == "ternary_quantile":
        spec["q_lo"] = float(valid.quantile(lower_q))
        spec["q_hi"] = float(valid.quantile(upper_q))
    return spec



def apply_label_spec(ret_series: pd.Series, spec: Dict[str, Any]) -> pd.Series:
    s = ret_series.copy()
    labels = pd.Series(np.nan, index=s.index, dtype=float)
    method = spec["method"]

    if method == "ternary_quantile":
        labels.loc[s.notna()] = 1
        labels.loc[s >= spec["q_hi"]] = 2
        labels.loc[s <= spec["q_lo"]] = 0
    elif method == "ternary_threshold":
        labels.loc[s.notna()] = 1
        labels.loc[s > spec["upper_thresh"]] = 2
        labels.loc[s < spec["lower_thresh"]] = 0
    elif method == "binary_sign":
        labels.loc[s.notna()] = (s[s.notna()] > 0).astype(float)
    else:
        raise ValueError(f"Unknown method: {method}")

    return labels.astype("Int64")



def prepare_feature_block(
    feat: pd.DataFrame,
    feature_cols: List[str],
    fill_values: Optional[pd.Series] = None,
) -> Tuple[pd.DataFrame, pd.Series]:
    x = feat[["trade_date"] + feature_cols].copy().sort_values("trade_date").reset_index(drop=True)
    x.loc[:, feature_cols] = x.loc[:, feature_cols].ffill()
    if fill_values is None:
        fill_values = x.loc[:, feature_cols].median(numeric_only=True)
    x.loc[:, feature_cols] = x.loc[:, feature_cols].fillna(fill_values)
    x = x.dropna(subset=feature_cols).reset_index(drop=True)
    return x, fill_values



def build_classifier(clf_type: str = "lgbm", n_classes: int = 3):
    if clf_type == "lgbm_tuned" and _HAS_LGBM:
        return LGBMClassifier(
            n_estimators=300,
            learning_rate=0.02,
            num_leaves=7,
            max_depth=3,
            min_child_samples=60,
            subsample=0.9,
            colsample_bytree=0.6,
            reg_alpha=0.5,
            reg_lambda=2.0,
            class_weight="balanced",
            random_state=42,
            verbose=-1,
        )
    if clf_type == "lgbm" and _HAS_LGBM:
        return LGBMClassifier(
            n_estimators=120,
            learning_rate=0.03,
            num_leaves=15,
            max_depth=3,
            min_child_samples=40,
            subsample=0.8,
            colsample_bytree=0.7,
            reg_alpha=0.2,
            reg_lambda=1.0,
            class_weight="balanced",
            random_state=42,
            verbose=-1,
        )
    if clf_type == "logistic":
        return LogisticRegression(max_iter=1000, C=0.3, random_state=42)
    return GradientBoostingClassifier(
        n_estimators=120,
        learning_rate=0.03,
        max_depth=2,
        subsample=0.8,
        random_state=42,
    )



def rolling_supervised_oos(
    feat_lagged: pd.DataFrame,
    ret_use: pd.DataFrame,
    feature_cols: List[str],
    start_date: int,
    end_date: int = TRAIN_END,
    train_days: int = 480,
    test_days: int = 60,
    step_days: int = 20,
    label_method: str = "ternary_quantile",
    upper_q: float = 0.67,
    lower_q: float = 0.33,
    upper_thresh: float = 0.006,
    lower_thresh: float = -0.003,
    clf_type: str = "lgbm",
) -> pd.DataFrame:
    valid_ret_dates = set(
        ret_use.loc[
            (ret_use["trade_date"] >= start_date) & (ret_use["trade_date"] <= end_date),
            "trade_date",
        ]
    )
    feat_all = feat_lagged[
        (feat_lagged["trade_date"] >= start_date)
        & (feat_lagged["trade_date"] <= end_date)
        & (feat_lagged["trade_date"].isin(valid_ret_dates))
    ].copy().reset_index(drop=True)

    dates = feat_all["trade_date"].drop_duplicates().sort_values().tolist()
    all_oos = []

    start_idx = train_days
    while start_idx < len(dates):
        train_dates = dates[start_idx - train_days:start_idx]
        test_dates = dates[start_idx:min(start_idx + test_days, len(dates))]

        feat_train = feat_all[feat_all["trade_date"].isin(train_dates)].copy()
        feat_test = feat_all[feat_all["trade_date"].isin(test_dates)].copy()
        if len(feat_train) < 100 or len(feat_test) == 0:
            start_idx += step_days
            continue

        x_train, fill_values = prepare_feature_block(feat_train, feature_cols)
        x_train = x_train.merge(ret_use, on="trade_date", how="inner")
        if len(x_train) < 100:
            start_idx += step_days
            continue

        label_spec = fit_label_spec(
            x_train["ret"],
            method=label_method,
            upper_q=upper_q,
            lower_q=lower_q,
            upper_thresh=upper_thresh,
            lower_thresh=lower_thresh,
        )
        y_train = apply_label_spec(x_train["ret"], label_spec)
        valid = y_train.notna()
        if valid.sum() < 30 or y_train[valid].nunique() < 2:
            start_idx += step_days
            continue

        scaler = StandardScaler()
        X_tr = scaler.fit_transform(x_train.loc[valid, feature_cols].values)
        y_tr = y_train.loc[valid].astype(int).values

        clf = build_classifier(clf_type, len(np.unique(y_tr)))
        clf.fit(X_tr, y_tr)

        x_test, _ = prepare_feature_block(feat_test, feature_cols, fill_values=fill_values)
        X_te = scaler.transform(x_test[feature_cols].values)
        pred = clf.predict(X_te)

        out = x_test[["trade_date"]].copy()
        out["state"] = pred
        out["train_end_date"] = train_dates[-1]
        out["test_start_date"] = test_dates[0]
        out["test_end_date"] = test_dates[-1]
        all_oos.append(out)
        start_idx += step_days

    if not all_oos:
        return pd.DataFrame()

    oos = pd.concat(all_oos, ignore_index=True)
    oos = oos.drop_duplicates(subset=["trade_date"], keep="last")
    oos = oos.merge(ret_use, on="trade_date", how="left")
    return oos.sort_values("trade_date").reset_index(drop=True)



def score_oos_result(oos: pd.DataFrame) -> Dict[str, float]:
    perf = summarize_strategy_by_state(oos, state_col="state", ret_col="ret")
    if len(perf) < 2:
        return {
            "n_states": len(perf),
            "min_state_frac": np.nan,
            "ret_spread": np.nan,
            "best_mean_ret": np.nan,
            "worst_mean_ret": np.nan,
            "has_negative_state": 0.0,
        }
    return {
        "n_states": len(perf),
        "min_state_frac": perf["count_pct"].min(),
        "ret_spread": perf["mean_ret"].max() - perf["mean_ret"].min(),
        "best_mean_ret": perf["mean_ret"].max(),
        "worst_mean_ret": perf["mean_ret"].min(),
        "has_negative_state": float((perf["mean_ret"] < 0).any()),
    }



def tune_supervised_configs(
    feat_lagged: pd.DataFrame,
    ret_use: pd.DataFrame,
    feature_cols: List[str],
    candidate_configs: List[Dict[str, Any]],
    start_date: int,
    end_date: int = TRAIN_END,
) -> pd.DataFrame:
    rows = []
    for cfg in candidate_configs:
        oos = rolling_supervised_oos(
            feat_lagged=feat_lagged,
            ret_use=ret_use,
            feature_cols=feature_cols,
            start_date=start_date,
            end_date=end_date,
            **cfg,
        )
        score = score_oos_result(oos)
        rows.append({
            **cfg,
            "oos_rows": len(oos),
            **score,
        })
    return (
        pd.DataFrame(rows)
        .sort_values(
            ["has_negative_state", "ret_spread", "best_mean_ret", "min_state_frac", "oos_rows"],
            ascending=[False, False, False, False, False],
        )
        .reset_index(drop=True)
    )



def fit_supervised_model(
    feat_lagged: pd.DataFrame,
    ret_use: pd.DataFrame,
    feature_cols: List[str],
    start_date: int,
    train_end: int = TRAIN_END,
    label_method: str = "ternary_quantile",
    upper_q: float = 0.67,
    lower_q: float = 0.33,
    upper_thresh: float = 0.006,
    lower_thresh: float = -0.003,
    clf_type: str = "lgbm",
) -> Dict[str, Any]:
    fit_feat = feat_lagged[
        (feat_lagged["trade_date"] >= start_date) & (feat_lagged["trade_date"] <= train_end)
    ].copy().reset_index(drop=True)
    fit_ret = ret_use[
        (ret_use["trade_date"] >= start_date) & (ret_use["trade_date"] <= train_end)
    ].copy().reset_index(drop=True)

    x_fit, fill_values = prepare_feature_block(fit_feat, feature_cols)
    x_fit = x_fit.merge(fit_ret, on="trade_date", how="inner")
    label_spec = fit_label_spec(
        x_fit["ret"],
        method=label_method,
        upper_q=upper_q,
        lower_q=lower_q,
        upper_thresh=upper_thresh,
        lower_thresh=lower_thresh,
    )
    y_fit = apply_label_spec(x_fit["ret"], label_spec)
    valid = y_fit.notna()
    if valid.sum() < 30 or y_fit[valid].nunique() < 2:
        raise ValueError("Not enough labeled rows to fit supervised model")

    scaler = StandardScaler()
    X_fit = scaler.fit_transform(x_fit.loc[valid, feature_cols].values)
    y = y_fit.loc[valid].astype(int).values

    clf = build_classifier(clf_type, len(np.unique(y)))
    clf.fit(X_fit, y)

    return {
        "model": clf,
        "scaler": scaler,
        "fill_values": fill_values,
        "label_spec": label_spec,
        "feature_cols": feature_cols,
        "train_frame": x_fit.loc[valid].copy().reset_index(drop=True),
        "y_train": y,
        "train_end": train_end,
        "last_label_date": int(ret_use["trade_date"].max()),
    }



def predict_full_sample(
    feat_lagged: pd.DataFrame,
    ret_use: pd.DataFrame,
    fit_pack: Dict[str, Any],
    start_date: int,
) -> pd.DataFrame:
    feature_cols = fit_pack["feature_cols"]
    full_feat = feat_lagged[feat_lagged["trade_date"] >= start_date].copy().reset_index(drop=True)
    x_full, _ = prepare_feature_block(full_feat, feature_cols, fill_values=fit_pack["fill_values"])

    X_full = fit_pack["scaler"].transform(x_full[feature_cols].values)
    pred = fit_pack["model"].predict(X_full)
    out = x_full[["trade_date"]].copy()
    out["state"] = pred.astype(int)
    out["state_name"] = out["state"].map({0: "bad", 1: "neutral", 2: "good"}).fillna("unknown")

    if hasattr(fit_pack["model"], "predict_proba"):
        prob = fit_pack["model"].predict_proba(X_full)
        for i in range(prob.shape[1]):
            out[f"prob_{i}"] = prob[:, i]

    out = out.merge(ret_use, on="trade_date", how="left")
    out["ret_label"] = apply_label_spec(out["ret"], fit_pack["label_spec"])
    out["has_label"] = out["ret"].notna()
    out["is_after_train_end"] = out["trade_date"] > fit_pack["train_end"]
    out["is_after_last_label_date"] = out["trade_date"] > fit_pack["last_label_date"]
    return out.sort_values("trade_date").reset_index(drop=True)



def feature_importance_table(fit_pack: Dict[str, Any]) -> pd.DataFrame:
    clf = fit_pack["model"]
    feature_cols = fit_pack["feature_cols"]
    if hasattr(clf, "feature_importances_"):
        imp = clf.feature_importances_
    elif hasattr(clf, "coef_"):
        coef = np.asarray(clf.coef_)
        imp = np.abs(coef).mean(axis=0)
    else:
        return pd.DataFrame()
    return pd.DataFrame({
        "feature": feature_cols,
        "importance": imp,
    }).sort_values("importance", ascending=False).reset_index(drop=True)


## 5. Load Data

In [ ]:

df_raw = pd.read_csv(INDEX_PATH)
df_raw["trade_date"] = df_raw["trade_date"].astype(int)

df_ret = pd.read_csv(RETURN_PATH)
df_ret["date"] = df_ret["date"].astype(int)

print(f"index rows = {len(df_raw):,}, date range = {df_raw['trade_date'].min()} ~ {df_raw['trade_date'].max()}")
print(f"return rows = {len(df_ret):,}, date range = {df_ret['date'].min()} ~ {df_ret['date'].max()}")


## 6. Build Feature Matrix & Resolve Usable Start Date


In [ ]:

df_ret_use = build_return_table(df_ret)
feat_base, base_feature_cols = build_feature_matrix(df_raw)
base_feature_start_info, index_start_info, base_model_start_date = summarize_feature_availability(feat_base, base_feature_cols)

baseline_fit = fit_supervised_model(
    feat_lagged=feat_base,
    ret_use=df_ret_use,
    feature_cols=base_feature_cols,
    start_date=base_model_start_date,
    train_end=TRAIN_END,
    label_method="ternary_quantile",
    upper_q=0.67,
    lower_q=0.33,
    clf_type="lgbm",
)
base_top_features = feature_importance_table(baseline_fit).head(20)["feature"].tolist()

feat_lagged, strategy_feature_cols = add_strategy_return_features(feat_base, df_ret_use)
feature_cols = [c for c in base_top_features + strategy_feature_cols if c in feat_lagged.columns]
feature_start_info, _, model_start_date = summarize_feature_availability(feat_lagged, feature_cols)

print(f"feature matrix rows = {len(feat_lagged):,}")
print(f"base feature count = {len(base_feature_cols)}")
print(f"selected feature count = {len(feature_cols)}")
print(f"train_end = {TRAIN_END}")
print(f"model_start_date = {model_start_date}")
print("top base features:", base_top_features)
print("strategy return features:", strategy_feature_cols)

display(index_start_info)


## 7. Fixed Best Configuration


In [ ]:
best_cfg = {
    "train_days": 540,
    "test_days": 40,
    "step_days": 20,
    "label_method": "ternary_threshold",
    "clf_type": "lgbm_tuned",
    "upper_thresh": 0.006,
    "lower_thresh": -0.003,
}
print(best_cfg)


## 8. Pure Out-of-Sample Evaluation on All Post-20241231 Labeled Dates


In [ ]:
fit_pack = fit_supervised_model(
    feat_lagged=feat_lagged,
    ret_use=df_ret_use,
    feature_cols=feature_cols,
    start_date=model_start_date,
    train_end=TRAIN_END,
    label_method=best_cfg["label_method"],
    upper_q=best_cfg.get("upper_q", 0.67),
    lower_q=best_cfg.get("lower_q", 0.33),
    upper_thresh=best_cfg.get("upper_thresh", 0.006),
    lower_thresh=best_cfg.get("lower_thresh", -0.003),
    clf_type=best_cfg["clf_type"],
)
full_pred = predict_full_sample(
    feat_lagged=feat_lagged,
    ret_use=df_ret_use,
    fit_pack=fit_pack,
    start_date=model_start_date,
)

validation_oos = full_pred[
    (full_pred["trade_date"] > TRAIN_END)
    & (full_pred["has_label"])
].copy().reset_index(drop=True)

print(f"training rows = {len(fit_pack['train_frame']):,}")
print(f"training window = {int(fit_pack['train_frame']['trade_date'].min())} ~ {int(fit_pack['train_frame']['trade_date'].max())}")
print(f"validation rows = {len(validation_oos):,}")
if len(validation_oos):
    print(
        "validation window = "
        f"{int(validation_oos['trade_date'].min())} ~ {int(validation_oos['trade_date'].max())}"
    )
    oos_eval = evaluate_oos_states(validation_oos, label=f"Post-{TRAIN_END} Pure OOS")
else:
    print("validation window = none")
    oos_eval = {}


In [ ]:

oos_with_labels = validation_oos.copy()
cm_df = oos_with_labels[["state", "ret_label"]].dropna().copy()
if len(cm_df):
    y_true = cm_df["ret_label"].astype(int)
    y_pred = cm_df["state"].astype(int)
    used_labels = sorted(set(y_true) | set(y_pred))
    print("confusion matrix")
    display(pd.DataFrame(
        confusion_matrix(y_true, y_pred, labels=used_labels),
        index=[f"true_{x}" for x in used_labels],
        columns=[f"pred_{x}" for x in used_labels],
    ))
    print(classification_report(y_true, y_pred, zero_division=0))


## 9. Training Sample Diagnostics


In [ ]:
print("model fitted on <= train_end only")
print("train_end:", fit_pack["train_end"])
print("last labeled date:", fit_pack["last_label_date"])
if len(validation_oos):
    print(
        "post-train labeled validation window:",
        int(validation_oos["trade_date"].min()),
        "~",
        int(validation_oos["trade_date"].max()),
    )
else:
    print("post-train labeled validation window: none")


In [ ]:

fi = feature_importance_table(fit_pack)
fi.head(25)


## 10. Full-Sample Prediction & Export


In [ ]:
# full_pred.to_csv(OUTPUT_PATH, index=False)
print(f"saved to: {OUTPUT_PATH}")
print(f"prediction rows = {len(full_pred):,}")
print(f"labeled rows = {int(full_pred['has_label'].sum()):,}")
print(f"post-train labeled OOS rows = {int(((full_pred['trade_date'] > TRAIN_END) & full_pred['has_label']).sum()):,}")
print(f"2026 labeled rows = {int(((full_pred['trade_date'] >= 20260101) & full_pred['has_label']).sum()):,}")
print(f"unlabeled future rows = {int((~full_pred['has_label']).sum()):,}")


In [ ]:
full_pred[['trade_date', 'state']].to_csv(OUTPUT_PATH, index=False)